Run before everything

In [ ]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

Part3

In [ ]:
import time

def part3run():
    print("Part 3 Started")
    target_side = ""

    # --- 1. Line Trace to the Junction ---
    print("Tracing to junction...")
    got.set_track_recognition_line(0) # Black line mode
    
    while True:
        offset, line_type, x, y = got.get_single_track_total_info()
        
        # If line_type indicates an intersection or a specific marker
        # (Usually line_type 2 or 3 for junctions, or we check for color)
        color_info = got.get_color_total_info()
        
        if color_info and color_info[0] in ["Red", "Green"]:
            # --- 2. Recognise Color at Junction (10 pts) ---
            if color_info[0] == "Red":
                target_side = "LEFT"
                got.screen_display_background(3)
            else:
                target_side = "RIGHT"
                got.screen_display_background(6)
            
            got.screen_print(f"TURN {target_side}")
            got.mecanum_stop()
            print(f"Color seen: {color_info[0]}. Turning {target_side}")
            break
        
        # Keep following the line until we see the color marker
        steering = int(offset * 0.25)
        got.mecanum_move_xyz(0, 25, steering)
        time.sleep(0.01)

    # --- 3. Move to Correct Scoring Zone (20 pts) ---
    if target_side == "LEFT":
        got.mecanum_turn_speed_times(turn=2, speed=30, times=45, unit=2)
    else:
        got.mecanum_turn_speed_times(turn=3, speed=30, times=45, unit=2)

    # Continue tracing to the final drop-off point
    while True:
        offset, line_type, x, y = got.get_single_track_total_info()
        if line_type == 0: # End of line = Scoring Zone
            got.mecanum_stop()
            break
        steering = int(offset * 0.25)
        got.mecanum_move_xyz(0, 20, steering)
        time.sleep(0.01)

    # --- 4. Place Token (40 pts) ---
    # Criterion: "Place token in correct scoring zone"
    print("Dropping token...")
    put_down() # Using your put_down function
    
    # Back up slightly so we don't hit the token
    got.mecanum_move_xyz(0, -15, 0)
    time.sleep(1.0)
    got.mecanum_stop()
    
    got.screen_print("MISSION COMPLETE")
    print("Part 3 Finished")

# Run Part 3
try:
    part3run()
except KeyboardInterrupt:
    got.mecanum_stop()